In [1]:
import os

BASE = "/kaggle/input/datasets/filipacalheiros"

DATA_ROOT = f"{BASE}/deepfashion-data"
CODE_ROOT = f"{BASE}/project-code"

print("DATA_ROOT:", DATA_ROOT)
print("CODE_ROOT:", CODE_ROOT)

print("DATA_ROOT exists:", os.path.exists(DATA_ROOT))
print("CODE_ROOT exists:", os.path.exists(CODE_ROOT))

print("DATA files:", os.listdir(DATA_ROOT))
print("CODE files:", os.listdir(CODE_ROOT))
print("Image folders example:", os.listdir(f"{DATA_ROOT}/img")[:5])


DATA_ROOT: /kaggle/input/datasets/filipacalheiros/deepfashion-data
CODE_ROOT: /kaggle/input/datasets/filipacalheiros/project-code
DATA_ROOT exists: True
CODE_ROOT exists: True
DATA files: ['selected_attributes.txt', 'metadata.csv', 'img']
CODE files: ['requirements.txt', 'src']
Image folders example: ['Slub_Knit_Cutout_Tee', 'Heathered_Cutout_Leggings', 'Collared_Surplice_Romper', 'Stonewash_-_Skinny_Jeans', 'Quilted_Faux_Leather_Skirt']


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


CUDA available: True
Device: Tesla T4


In [3]:
!pip install -q pandas scikit-learn pillow tqdm


In [4]:
!python "$CODE_ROOT/src/train.py" \
  --metadata "$DATA_ROOT/metadata.csv" \
  --data-root "$DATA_ROOT" \
  --epochs 1 \
  --batch-size 8 \
  --image-size 128 \
  --max-train-batches 2 \
  --max-val-batches 2 \
  --no-pretrained \
  --output-dir /kaggle/working/debug_checkpoints


Device: cuda
Epoch 001/001 train_loss=1.7124 val_loss=2.0562 f1_micro=0.0392 f1_macro=0.0245 
Saved best checkpoint: /kaggle/working/debug_checkpoints/best_resnet50.pt


In [5]:
import os
import pandas as pd

metadata_path = f"{DATA_ROOT}/metadata.csv"
df = pd.read_csv(metadata_path)

existing = set()

for dirpath, _, filenames in os.walk(f"{DATA_ROOT}/img"):
    for filename in filenames:
        rel = os.path.relpath(os.path.join(dirpath, filename), DATA_ROOT)
        existing.add(rel.replace("\\", "/"))

exists_mask = df["image_path"].isin(existing)

print("Total rows:", len(df))
print("Existing images:", int(exists_mask.sum()))
print("Missing images:", int((~exists_mask).sum()))

print("Missing examples:")
print(df.loc[~exists_mask, "image_path"].head(10).to_string(index=False))

filtered_path = "/kaggle/working/metadata_filtered.csv"
df.loc[exists_mask].to_csv(filtered_path, index=False)

print("Saved:", filtered_path)


Total rows: 289222
Existing images: 289147
Missing images: 75
Missing examples:
img/Striped_A-line_Dress/img_00000001.jpg
img/Striped_A-line_Dress/img_00000002.jpg
img/Striped_A-line_Dress/img_00000003.jpg
img/Striped_A-line_Dress/img_00000004.jpg
img/Striped_A-line_Dress/img_00000005.jpg
img/Striped_A-line_Dress/img_00000006.jpg
img/Striped_A-line_Dress/img_00000007.jpg
img/Striped_A-line_Dress/img_00000008.jpg
img/Striped_A-line_Dress/img_00000009.jpg
img/Striped_A-line_Dress/img_00000010.jpg
Saved: /kaggle/working/metadata_filtered.csv


In [8]:
!python "$CODE_ROOT/src/train.py" \
  --metadata /kaggle/working/metadata_filtered.csv \
  --data-root "$DATA_ROOT" \
  --epochs 3 \
  --batch-size 32 \
  --image-size 224 \
  --max-train-batches 3000 \
  --max-val-batches 500 \
  --lr 1e-4 \
  --output-dir /kaggle/working/checkpoints


Device: cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 163MB/s]
Epoch 001/003 train_loss=0.9488 val_loss=0.8255 f1_micro=0.1732 f1_macro=0.1577 
Saved best checkpoint: /kaggle/working/checkpoints/best_resnet50.pt
Epoch 002/003 train_loss=0.8305 val_loss=0.7924 f1_micro=0.1846 f1_macro=0.1622 
Saved best checkpoint: /kaggle/working/checkpoints/best_resnet50.pt
Epoch 003/003 train_loss=0.7809 val_loss=0.7793 f1_micro=0.1927 f1_macro=0.1721 
Saved best checkpoint: /kaggle/working/checkpoints/best_resnet50.pt


In [9]:
!python "$CODE_ROOT/src/infer.py" \
  --checkpoint /kaggle/working/checkpoints/best_resnet50.pt \
  --data-root "$DATA_ROOT" \
  --image "img/Sheer_Pleated-Front_Blouse/img_00000001.jpg" \
  --top-k 10


Image: img/Sheer_Pleated-Front_Blouse/img_00000001.jpg
long sleeve: 0.9296
chiffon: 0.8978
sleeve: 0.8729
sheer: 0.8002
pleated: 0.7361
collar: 0.7013
v neck: 0.6640
shirt: 0.6549
woven: 0.6090
button: 0.5956


In [10]:
import shutil
from pathlib import Path

src = Path("/kaggle/working/checkpoints/best_resnet50.pt")
dst = Path("/kaggle/working/best_resnet50.pt")

shutil.copy2(src, dst)

print("Saved for download:", dst)


Saved for download: /kaggle/working/best_resnet50.pt
